## Krippendorff Alphs from Human Annotators and LLM

In [1]:
import pandas as pd
import numpy as np

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
scores = pd.read_csv('human_and_llm_scores.csv')

In [5]:
scores.head()

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,caden,...,p05,p06,p07,p08,p09,p10,p13,p16,p17,llm_score
0,0,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.31,5,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,1.0,...,2.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,2
1,1,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.66,4,Psalter,43,"We have heard with our ears, O God, for our fa...",0.0,...,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
2,2,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.69,3,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",1.0,...,3.0,2.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,1
3,3,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",3.0,...,NaN,1.0,NaN,2.0,NaN,1.0,NaN,NaN,NaN,1
4,4,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,1.0,...,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2


In [6]:
%pip install krippendorff

Note: you may need to restart the kernel to use updated packages.


In [7]:
import krippendorff

scores.columns

Index(['Unnamed: 0', 'Query', 'Query Category', 'Method',
       'Similarity Score (%)', 'Rank', 'Text', 'Psalm Num', 'Verse', 'caden',
       'p01', 'p02', 'p03', 'p04', 'p05', 'p06', 'p07', 'p08', 'p09', 'p10',
       'p13', 'p16', 'p17', 'llm_score'],
      dtype='str')

In [9]:
annotator_cols = ['caden', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06',
                  'p07', 'p08', 'p09', 'p10', 'p13', 'p16', 'p17', 'llm_score']

data = scores[annotator_cols].to_numpy().T

alpha = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='ordinal'
)

print(f"Krippendorff's Alpha: {alpha:.4f}")

Krippendorff's Alpha: 0.1592


In [28]:
import krippendorff

# -------------------------
# 1. Everyone together
# -------------------------
annotator_cols = [
    'caden', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06',
    'p07', 'p08', 'p09', 'p10', 'p13', 'p16', 'p17', 'llm_score'
]

data = scores[annotator_cols].to_numpy().T

alpha_all = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='ordinal'
)

print(f"All Annotators + LLM: {alpha_all:.4f}")


# -------------------------
# 2. Caden + LLM
# -------------------------
caden_llm_cols = ['caden', 'llm_score']

data = scores[caden_llm_cols].to_numpy().T

alpha_caden_llm = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='ordinal'
)

print(f"Caden + LLM:          {alpha_caden_llm:.4f}")


# -------------------------
# 3. Other humans + LLM
# -------------------------
other_humans_llm_cols = [
    'p01', 'p02', 'p03', 'p04', 'p05', 'p06',
    'p07', 'p08', 'p09', 'p10', 'p13', 'p16', 'p17',
    'llm_score'
]

data = scores[other_humans_llm_cols].to_numpy().T

alpha_other_humans_llm = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='ordinal'
)

print(f"Other Humans + LLM:   {alpha_other_humans_llm:.4f}")

All Annotators + LLM: 0.1592
Caden + LLM:          -0.0497
Other Humans + LLM:   0.1605


In [29]:
import pandas as pd
import krippendorff

human_cols = [
    'caden', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06',
    'p07', 'p08', 'p09', 'p10', 'p13', 'p16', 'p17'
]

results = []

for annotator in human_cols:
    data = scores[[annotator, 'llm_score']].to_numpy().T

    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement='ordinal'
    )

    results.append({
        'Annotator': annotator,
        'Krippendorff Alpha': alpha
    })

alpha_results = pd.DataFrame(results)

display(alpha_results)

,Annotator,Krippendorff Alpha
0,caden,-0.049666
1,p01,-0.063966
2,p02,0.007283
3,p03,-0.058617
4,p04,0.022049
5,p05,-0.139271
6,p06,-0.032417
7,p07,-0.006121
8,p08,0.043459
9,p09,-0.286746


## Cheecking agreeement possibilities 

In [16]:
pd.set_option('display.max_columns', None)


In [19]:
# True if any value appears more than once in the row
scores['any_agreement'] = scores[annotator_cols].nunique(axis=1) < len(annotator_cols)
# scores

In [22]:
scores['caden_llm_agree'] = (
    scores['caden'] == scores['llm_score']
)

In [24]:
other_humans = [
    'p01', 'p02', 'p03', 'p04', 'p05', 'p06',
    'p07', 'p08', 'p09', 'p10', 'p13', 'p16', 'p17'
]

scores['any_human_llm_agree'] = (
    scores[other_humans]
    .eq(scores['llm_score'], axis=0)
    .any(axis=1)
)

In [26]:
cols = [
    'any_agreement',
    'caden_llm_agree',
    'any_human_llm_agree'
]

comparison = pd.DataFrame({
    col: scores[col].value_counts()
    for col in cols
}).fillna(0).astype(int)

display(comparison)

,any_agreement,caden_llm_agree,any_human_llm_agree
False,0,152,92
True,193,41,101


In [27]:
summary = pd.DataFrame({
    'Count': scores[cols].sum(),
    'Percent': scores[cols].mean() * 100
})

display(summary)

,Count,Percent
any_agreement,193,100.000000
caden_llm_agree,41,21.243523
any_human_llm_agree,101,52.331606


In [25]:
scores

,Unnamed: 0,Query,Query Category,Method,Similarity Score (%),Rank,Text,Psalm Num,Verse,caden,p01,p02,p03,p04,p05,p06,p07,p08,p09,p10,p13,p16,p17,llm_score,any_agreement,agrees_with_caden,caden_llm_agree,any_human_llm_agree
0,0,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.31,5,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,1.0,NaN,NaN,0.0,NaN,2.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,2,True,False,False,True
1,1,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.66,4,Psalter,43,"We have heard with our ears, O God, for our fa...",0.0,1.0,NaN,NaN,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,True,False,False,False
2,2,Create in me a clean heart,Phrase/Exact Match Queries,BERT,68.69,3,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",1.0,NaN,NaN,NaN,NaN,3.0,2.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,1,True,True,True,False
3,3,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.06,2,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",3.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,2.0,NaN,1.0,NaN,NaN,NaN,1,True,False,False,True
4,4,Create in me a clean heart,Phrase/Exact Match Queries,BERT,69.89,1,Bible,18,For the End a psalm by David The heavens decla...,1.0,NaN,3.0,0.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,188,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,18.97,5,Bible,151,1This is a psalm written with Davids own hand ...,2.0,1.0,3.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,True,False,False,True
189,189,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,22.61,4,Psalter,28,"Bring unto the Lord, O ye sons of God, bring u...",1.0,NaN,NaN,NaN,NaN,NaN,0.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,1,True,True,True,False
190,190,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,23.85,3,Psalter,86,His foundations are in the holy mountains. The...,1.0,NaN,NaN,NaN,1.0,NaN,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,3,True,True,False,False
191,191,protection from enemies,Thematic/Semantic Queries,TFIDF_GLoVe,25.01,2,Bible,28,A psalm by David the final day of the Feast of...,3.0,1.0,NaN,NaN,NaN,NaN,0.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3,True,True,True,True


In [18]:
scores['any_agreement'].value_counts()

any_agreement
True    193
Name: count, dtype: int64

* An LLM judge may produce useful aggregate evaluations while exhibiting poor agreement with individual human judgments.
* That's particularly relevant because your task involves subjective semantic/religious relevance judgments.

In [30]:
import pandas as pd
from scipy.stats import kendalltau

human_cols = [
    'caden', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06',
    'p07', 'p08', 'p09', 'p10', 'p13', 'p16', 'p17'
]

results = []

for annotator in human_cols:
    tau, p_value = kendalltau(
        scores[annotator],
        scores['llm_score']
    )

    results.append({
        'Annotator': annotator,
        "Kendall's Tau": tau,
        'p-value': p_value
    })

kendall_results = pd.DataFrame(results)

display(kendall_results)

,Annotator,Kendall's Tau,p-value
0,caden,-0.014764,0.807685
1,p01,NaN,NaN
2,p02,NaN,NaN
3,p03,NaN,NaN
4,p04,NaN,NaN
5,p05,NaN,NaN
6,p06,NaN,NaN
7,p07,NaN,NaN
8,p08,NaN,NaN
9,p09,NaN,NaN


In [31]:
human_cols = [
    'caden', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06',
    'p07', 'p08', 'p09', 'p10', 'p13', 'p16', 'p17'
]

scores['human_mean'] = scores[human_cols].mean(axis=1)

tau, p_value = kendalltau(
    scores['human_mean'],
    scores['llm_score']
)

print(f"Kendall's τ: {tau:.4f}")
print(f"p-value:      {p_value:.4f}")

Kendall's τ: 0.0086
p-value:      0.8776


| Analysis                       |         Your result | Interpretation                |
| ------------------------------ | ------------------: | ----------------------------- |
| Humans + LLM Krippendorff's α  |          **0.1592** | Low overall reliability       |
| Individual human–LLM α         | **−0.472 to 0.043** | Very low agreement            |
| Mean human vs. LLM Kendall's τ |          **0.0086** | Essentially no association    |
| Kendall's τ p-value            |          **0.8776** | Not statistically significant |

| **Agreement Measure** | **No Agreement** | **Agreement** | **Total** | **Agreement (%)** |
|---|---:|---:|---:|---:|
| Any human annotators agree | 0 | 193 | 193 | **100.0%** |
| Caden–LLM agreement | 152 | 41 | 193 | **21.2%** |
| Other human–LLM agreement | 92 | 101 | 193 | **52.3%** |


* human eval had as high cost and limited coverage
* llm eval 
    - low human agreement on an indivual level but has a greater coverage of results and conclusions\
    - picking up on contextual things that might be over looked by humans 
* Therefore, 
    - llm can supplement human evaluation and provide scalable aggregate analysis, rather than fully replacing human judgement.

# Score Stats

In [42]:
summary = pd.DataFrame({
    'Average': scores[columns].mean(),
    'Count': scores[columns].count()
})

summary = summary.reset_index().rename(columns={'index': 'User'})

# Split into 3 roughly equal DataFrames
n = len(summary)
chunk_size = (n + 2) // 3  # ceiling division

parts = [
    summary.iloc[i:i + chunk_size].reset_index(drop=True)
    for i in range(0, n, chunk_size)
]

# Ensure exactly 3 parts
while len(parts) < 3:
    parts.append(pd.DataFrame(columns=summary.columns))

wide_summary = pd.concat(parts, axis=1)

wide_summary.columns = [
    'User', 'Average', 'Count',
    'User', 'Average', 'Count',
    'User', 'Average', 'Count'
]

print(
    wide_summary.to_latex(
        index=False,
        escape=False,
        float_format="%.2f",
        column_format="lrl lrl lrl"
    )
)

\begin{tabular}{lrl lrl lrl}
\toprule
User & Average & Count & User & Average & Count & User & Average & Count \\
\midrule
caden & 1.82 & 193 & p05 & 2.64 & 76 & p10 & 0.72 & 61 \\
p01 & 1.44 & 85 & p06 & 1.27 & 89 & p13 & 2.50 & 20 \\
p02 & 1.70 & 27 & p07 & 2.10 & 20 & p16 & 2.86 & 7 \\
p03 & 0.87 & 85 & p08 & 1.98 & 53 & p17 & 0.33 & 3 \\
p04 & 1.92 & 25 & p09 & 1.50 & 28 & llm_score & 1.45 & 193 \\
\bottomrule
\end{tabular}



\begin{tabular}{lrl lrl lrl}
\toprule
User & Average & Count & User & Average & Count & User & Average & Count \\
\midrule
caden & 1.82 & 193 & p05 & 2.64 & 76 & p10 & 0.72 & 61 \\
p01 & 1.44 & 85 & p06 & 1.27 & 89 & p13 & 2.50 & 20 \\
p02 & 1.70 & 27 & p07 & 2.10 & 20 & p16 & 2.86 & 7 \\
p03 & 0.87 & 85 & p08 & 1.98 & 53 & p17 & 0.33 & 3 \\
p04 & 1.92 & 25 & p09 & 1.50 & 28 & llm_score & 1.45 & 193 \\
\bottomrule
\end{tabular}

In [39]:
headers = [
    r"\textbf{User}", r"\textbf{Average}", r"\textbf{Count}",
    r"\textbf{User}", r"\textbf{Average}", r"\textbf{Count}",
    r"\textbf{User}", r"\textbf{Average}", r"\textbf{Count}"
]

latex_table = wide_summary.to_latex(
    index=False,
    header=headers,
    escape=False,
    float_format="%.2f",
    column_format="lrllrllrl"
)

print(latex_table)

ValueError: Writing 6 cols but got 9 aliases